# MintyPython Demo

This notebook demonstrates all major features of MintyPython (Model INTerpretation with pYthon) - a visualization library for comparing GLM and GBM models.

**Contents:**
1. Setup & Data Generation
2. Initialize MintyPython
3. SHAP Value Computation
4. Univariate Plots
5. Bivariate Plots
6. Model Comparison
7. Configuration Customization
8. Plot Engines (Bokeh vs Matplotlib)

## 1. Setup & Data Generation

First, let's import the required packages and generate synthetic insurance data.

In [1]:
# Standard imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Bokeh setup for notebook display
from bokeh.io import output_notebook
output_notebook()

# Import demo modules
from synthetic_data import generate_synthetic_data, train_xgboost_model, prepare_data_for_mintypython
from glm_helpers import get_glm_data_for_mintypython

# Import MintyPython
from mintypython import mintypython

print("All imports successful!")

Loading BokehJS ...

All imports successful!


In [2]:
# Generate synthetic insurance data
raw_data = generate_synthetic_data(n_samples=10000, random_state=42)

print(f"Generated {len(raw_data):,} samples")
print(f"\nColumns: {list(raw_data.columns)}")
raw_data.head()

Generated 10,000 samples

Columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type', 'exposure', 'claim_count']


,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count
0,52.450712,15689.588755,11.677637,North,SUV,0.639255,0
1,42.926035,18906.296023,24.677705,West,Sedan,0.946568,0
2,54.715328,16338.988603,31.575657,South,SUV,0.989004,0
3,67.845448,23276.720582,45.267958,South,Truck,0.839560,0
4,41.487699,40078.257759,18.461473,West,SUV,0.985150,0


In [3]:
raw_data.dtypes


age               float64
vehicle_value     float64
years_licensed    float64
region                str
vehicle_type          str
exposure          float64
claim_count         int64
dtype: object

In [4]:
# Define feature names (these are the ORIGINAL column names, used for plotting)
feature_names = ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']

# Prepare data for modeling:
# - Original columns keep human-readable values (for GLM and plot labels)
# - New {col}_encoded columns contain numeric codes (for GBM/XGBoost)
data, category_mappings = prepare_data_for_mintypython(raw_data, feature_names, optimize=True)

print("=" * 70)
print("KEY CONCEPT: GBM vs GLM use different column formats")
print("=" * 70)

print("\n📊 Category mappings (code -> label):")
for col, mapping in category_mappings.items():
    print(f"  {col}: {mapping}")

print("\n" + "-" * 70)
print("GBM (XGBoost) uses ENCODED columns with numeric codes:")
print("-" * 70)
print(data[['region_encoded', 'vehicle_type_encoded']].head())

print("\n" + "-" * 70)
print("GLM uses ORIGINAL columns with string labels:")
print("-" * 70)
print(data[['region', 'vehicle_type']].head())

print("\n✅ MintyPython handles the mapping automatically via category_mappings")

Encoded 2 categorical column(s): ['region', 'vehicle_type']
Encoded columns created: ['region_encoded', 'vehicle_type_encoded']
Optimized 4 numeric column(s):
  age: float64 -> float32
  years_licensed: float64 -> float32
  exposure: float64 -> float32
  claim_count: int64 -> int8
Memory: 635.4KB -> 469.4KB (26.1% reduction)
KEY CONCEPT: GBM vs GLM use different column formats

📊 Category mappings (code -> label):
  region: {0: 'East', 1: 'North', 2: 'South', 3: 'West'}
  vehicle_type: {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}

----------------------------------------------------------------------
GBM (XGBoost) uses ENCODED columns with numeric codes:
----------------------------------------------------------------------
   region_encoded  vehicle_type_encoded
0               1                     0
1               3                     1
2               2                     0
3               2                     3
4               3                     0

----------------------

In [5]:
# Show the complete data structure
print("Complete DataFrame structure:")
print(f"Columns: {list(data.columns)}")
print(f"\nData types:")
print(data.dtypes)
print(f"\nSample of categorical columns (original vs encoded):")
data[['region', 'region_encoded', 'vehicle_type', 'vehicle_type_encoded']].head(10)

Complete DataFrame structure:
Columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type', 'exposure', 'claim_count', 'region_encoded', 'vehicle_type_encoded']

Data types:
age                     float32
vehicle_value           float64
years_licensed          float32
region                      str
vehicle_type                str
exposure                float32
claim_count                int8
region_encoded             int8
vehicle_type_encoded       int8
dtype: object

Sample of categorical columns (original vs encoded):


,region,region_encoded,vehicle_type,vehicle_type_encoded
0,North,1,SUV,0
1,West,3,Sedan,1
2,South,2,SUV,0
3,South,2,Truck,3
4,West,3,SUV,0
5,South,2,Sedan,1
6,West,3,Sedan,1
7,West,3,Truck,3
8,North,1,Sedan,1
9,North,1,Sedan,1


In [6]:
# Train XGBoost model with Poisson objective
# NOTE: train_xgboost_model automatically uses _encoded columns for categoricals
model = train_xgboost_model(
    data,
    feature_names,  # Pass original names - function uses _encoded columns internally
    weight_col='exposure',
    target_col='claim_count',
    random_state=42,
    max_depth=4,
    eta=0.1
)

print("=" * 70)
print("XGBoost Model Training")
print("=" * 70)
print("\nModel uses ENCODED columns internally:")
print("  - region_encoded (0, 1, 2, 3) instead of ('East', 'North', 'South', 'West')")
print("  - vehicle_type_encoded (0, 1, 2, 3) instead of ('SUV', 'Sedan', 'Sports', 'Truck')")
print("\nFeature importance:")
importance = model.get_score(importance_type='gain')
for feat, score in sorted(importance.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {score:.2f}")

XGBoost Model Training

Model uses ENCODED columns internally:
  - region_encoded (0, 1, 2, 3) instead of ('East', 'North', 'South', 'West')
  - vehicle_type_encoded (0, 1, 2, 3) instead of ('SUV', 'Sedan', 'Sports', 'Truck')

Feature importance:
  age: 2.06
  years_licensed: 1.42
  vehicle_value: 1.32
  vehicle_type: 1.15
  region: 1.05


## 2. Initialize MintyPython

Create a MintyPython instance with our model and data. We'll also set up simulated GLM relativities for comparison.

In [7]:
# Create simulated GLM relativities and predictions
# NOTE: GLM uses the ENCODED columns internally but maps back to original labels
glm_df, glm_preds = get_glm_data_for_mintypython(data, feature_names)

# Add GLM predictions to data
data['glm_predictions'] = glm_preds

print("=" * 70)
print("GLM Relativities (simulated from Emblem-style model)")
print("=" * 70)
print(f"\nGLM columns: {list(glm_df.columns)}")
print(f"GLM predictions - Mean: {glm_preds.mean():.4f}, Actual Mean: {data['claim_count'].mean():.4f}")

print("\n" + "-" * 70)
print("GLM relativities for categorical variables (using encoded values):")
print("-" * 70)
print(f"Region unique GLM values: {glm_df['region'].unique()}")
print(f"Vehicle Type unique GLM values: {glm_df['vehicle_type'].unique()}")

GLM Relativities (simulated from Emblem-style model)

GLM columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']
GLM predictions - Mean: 0.0894, Actual Mean: 0.0489

----------------------------------------------------------------------
GLM relativities for categorical variables (using encoded values):
----------------------------------------------------------------------
Region unique GLM values: [ 0.    0.05  0.1  -0.05]
Vehicle Type unique GLM values: [ 0.1   0.   -0.05  0.4 ]


In [8]:
glm_df.head()

,age,vehicle_value,years_licensed,region,vehicle_type
0,-0.026164,-0.051856,-0.030197,0.00,0.10
1,0.031476,-0.036936,-0.264199,0.05,0.00
2,-0.039869,-0.048611,-0.388362,0.10,0.10
3,-0.119329,-0.020299,-0.634823,0.10,-0.05
4,0.040181,0.023171,-0.152307,0.05,0.10


In [9]:
# Initialize MintyPython with category_mappings to bridge GBM and GLM
# 
# KEY: category_mappings tells MintyPython how to:
# 1. Use _encoded columns for GBM SHAP calculations
# 2. Map numeric codes back to labels for plotting
# 3. Match GLM relativities with GBM features

print("=" * 70)
print("Initializing MintyPython with category_mappings")
print("=" * 70)

mp = mintypython(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,           # Original names (not _encoded)
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    category_mappings=category_mappings,   # Maps codes <-> labels
    verbose=True
)

print("\n" + "-" * 70)
print("Verification:")
print("-" * 70)
print(f"category_mappings stored: {mp.category_mappings}")
print(f"fac_mapping (for plot labels): {mp.fac_mapping}")

Initializing MintyPython with category_mappings
Processed 2 categorical column(s): ['region', 'vehicle_type']

----------------------------------------------------------------------
Verification:
----------------------------------------------------------------------
category_mappings stored: {'region': {0: 'East', 1: 'North', 2: 'South', 3: 'West'}, 'vehicle_type': {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}}
fac_mapping (for plot labels): {'region': {0: 'East', 1: 'North', 2: 'South', 3: 'West'}, 'vehicle_type': {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}}


In [10]:
data.head()

,age,vehicle_value,years_licensed,region,vehicle_type,exposure,claim_count,region_encoded,vehicle_type_encoded,glm_predictions
0,52.450714,15689.588755,11.677636,North,SUV,0.639255,0.0,1,0,0.076083
1,42.926037,18906.296023,24.677706,West,Sedan,0.946568,0.0,3,1,0.091188
2,54.715328,16338.988603,31.575657,South,SUV,0.989004,0.0,2,0,0.089980
3,67.845451,23276.720582,45.267960,South,Truck,0.839560,0.0,2,3,0.048821
4,41.487698,40078.257759,18.461473,West,SUV,0.985150,0.0,3,0,0.125659


## 3. SHAP Value Computation

MintyPython computes SHAP values lazily when needed. Let's precompute them to see the process.

In [11]:
# Compute SHAP values (this is also done automatically when plotting)
mp.Data_prep.prep_shap_values()

print("SHAP values computed and cached.")
print(f"\nSHAP DataFrame shape: {mp.shap_df.shape}")
print(f"SHAP columns: {list(mp.shap_df.columns)}")

SHAP values computed and cached.

SHAP DataFrame shape: (10000, 5)
SHAP columns: ['age', 'vehicle_value', 'years_licensed', 'region', 'vehicle_type']


## 4. Univariate Plots

Univariate plots show how individual features affect model predictions.

### 4.1 Basic Univariate with SHAP

In [12]:
# Basic univariate plot showing SHAP values for age
mp.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Age - Basic SHAP Plot'
)

Loading BokehJS ...

figure(id='p1002', ...)

### 4.2 SHAP with Points and Standard Deviation

In [13]:
# SHAP with individual points and standard deviation bands
mp.univariate_plot(
    var_name='age',
    shap=True,
    shap_points=True,
    shap_sd=True,
    n_shap_points=500,
    weight=True,
    plot_name='Age - SHAP with Points and SD'
)

figure(id='p1097', ...)

### 4.3 SHAP with GLM Relativities Overlay

In [14]:
# Compare GBM SHAP values with GLM relativities
mp.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    weight=True,
    plot_name='Age - SHAP vs GLM Comparison'
)

figure(id='p1207', ...)

### 4.4 Full Plot with Actuals

In [15]:
# Complete plot with SHAP, GLM, actuals, and weights
mp.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Complete Analysis'
)

figure(id='p1326', ...)

### 4.5 Custom Binning

In [16]:
# Custom binning with start, finish, and stepsize
mp.univariate_plot(
    var_name='age',
    shap=True,
    weight=True,
    start=20,
    finish=70,
    stepsize=5,
    infinity_lower=True,
    infinity_higher=True,
    plot_name='Age - Custom Binning (20-70, step=5)'
)

figure(id='p1469', ...)

In [17]:
# Custom binning with nlevels (automatic step calculation)
mp.univariate_plot(
    var_name='vehicle_value',
    shap=True,
    glm=True,
    weight=True,
    #glmindic_cols=['vehicle_value'],
    nlevels=8,
    percentile_start=5,
    percentile_finish=95,
    plot_name='Vehicle Value - 8 Levels with Percentile Bounds'
)

figure(id='p1564', ...)

### 4.6 Categorical Variables - GBM vs GLM Comparison

This is where the encoding really matters:
- **GBM SHAP**: Computed using `region_encoded` (numeric codes), then mapped back to labels
- **GLM Relativities**: Computed using `region_encoded`, displayed with string labels
- **X-axis labels**: Show human-readable values from `category_mappings`

In [18]:
# Region - categorical variable
# X-axis shows: East, North, South, West (from category_mappings)
# GBM internally uses: 0, 1, 2, 3 (from region_encoded)
# GLM relativities are mapped to the same labels

print("Data check:")
print(f"  data['region'] dtype: {data['region'].dtype} (string labels for plotting)")
print(f"  data['region_encoded'] dtype: {data['region_encoded'].dtype} (numeric codes for GBM)")
print()

mp.univariate_plot(
    var_name='region',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Region - GBM (SHAP) vs GLM Comparison'
)

Data check:
  data['region'] dtype: str (string labels for plotting)
  data['region_encoded'] dtype: int8 (numeric codes for GBM)



figure(id='p1683', ...)

In [19]:
# Vehicle type - categorical variable  
# Same principle: GBM uses encoded, plots show labels

print("Data check:")
print(f"  data['vehicle_type'] dtype: {data['vehicle_type'].dtype} (string labels)")
print(f"  data['vehicle_type_encoded'] dtype: {data['vehicle_type_encoded'].dtype} (numeric codes)")
print(f"  Mapping: {category_mappings['vehicle_type']}")
print()

mp.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Vehicle Type - GBM (SHAP) vs GLM Comparison'
)

Data check:
  data['vehicle_type'] dtype: str (string labels)
  data['vehicle_type_encoded'] dtype: int8 (numeric codes)
  Mapping: {0: 'SUV', 1: 'Sedan', 2: 'Sports', 3: 'Truck'}



figure(id='p1826', ...)

### 4.7 Rebasing to a Specific Level

In [20]:
# Set a specific base level for relativities
mp.univariate_plot(
    var_name='vehicle_type',
    shap=True,
    glm=True,
    weight=True,
    base='Sedan',  # Use Sedan (its encoded but it reuses the original labels)) as base
    rebase=True,
    plot_name='Vehicle Type - Rebased to Sedan'
)

figure(id='p1969', ...)

## 5. Bivariate Plots

Bivariate plots show interactions between two features.

### 5.1 Two Continuous Variables

In [21]:
# Age x Vehicle Value interaction
mp.bivariate_plot(
    var1='age',
    var2='vehicle_value',
    shap=True,
    nlevels_var1=6,
    nlevels_var2=4,
    plot_title='Age x Vehicle Value Interaction'
)

figure(id='p2088', ...)

### 5.2 Continuous x Categorical

In [22]:
# Age x Region interaction
mp.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    nlevels_var1=8,
    plot_title='Age x Region Interaction'
)

figure(id='p2263', ...)

In [23]:
# Vehicle Value x Vehicle Type interaction
mp.bivariate_plot(
    var1='vehicle_value',
    var2='vehicle_type',
    shap=True,
    nlevels_var1=6,
    plot_title='Vehicle Value x Vehicle Type Interaction'
)

figure(id='p2414', ...)

### 5.3 Bivariate with GLM

In [24]:
# Age x Region with GLM comparison
mp.bivariate_plot(
    var1='age',
    var2='region',
    shap=True,
    glm=True,
    nlevels_var1=6,
    plot_title='Age x Region - SHAP vs GLM'
)

figure(id='p2565', ...)

## 6. Model Comparison

Compare multiple models using the `compare()` method.

In [25]:
# Train a second model with different hyperparameters
model2 = train_xgboost_model(
    data,
    feature_names,
    weight_col='exposure',
    target_col='claim_count',
    random_state=123,
    max_depth=6,  # Deeper trees
    eta=0.05      # Lower learning rate
)

print("Second model trained!")

Second model trained!


In [26]:
# Create second MintyPython instance with category_mappings
mp2 = mintypython(
    data=data,
    model=model2,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    category_mappings=category_mappings,  # Use category_mappings, not mapping_dict
    verbose=False
)

print("Second MintyPython instance created!")

Second MintyPython instance created!


In [27]:
# Compare models on age variable
mp.compare(
    mintylist=[mp2],
    mintynames=['Model 1 (depth=4)', 'Model 2 (depth=6)'],
    var_name='age',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Age'
)

KeyError: 'shap@Model 2 (depth=6)'

In [ ]:
# Compare models on vehicle_type
mp.compare(
    mintylist=[mp2],
    mintynames=['Model 1', 'Model 2'],
    var_name='vehicle_type',
    shap=True,
    weight=True,
    plot_name='Model Comparison - Vehicle Type'
)

KeyError: 'shap@Model 2'

## 7. Configuration Customization

Customize plot appearance through the `config` dictionary.

In [ ]:
# View default configuration
print("Default configuration:")
import json
print(json.dumps(mp.config, indent=2))

In [ ]:
# Create a copy of mp with custom colors
mp_custom = mintypython(
    data=data,
    model=model,
    weight_col='exposure',
    actuals_col='claim_count',
    feature_names=feature_names,
    link_fn='poisson',
    glm_preds_col='glm_predictions',
    glm_df=glm_df,
    category_mappings=category_mappings,  # Use category_mappings
    shap_df=mp.shap_df,  # Reuse computed SHAP values
    verbose=False
)

# Customize colors
mp_custom.config['colors']['shap'] = '#e41a1c'       # Red
mp_custom.config['colors']['glm'] = '#4daf4a'        # Green
mp_custom.config['colors']['weight'] = '#984ea3'     # Purple
mp_custom.config['colors']['actuals'] = '#ff7f00'    # Orange

# Customize labels
mp_custom.config['labels']['shap'] = 'GBM Effect'
mp_custom.config['labels']['glm'] = 'GLM Factor'

# Customize line widths
mp_custom.config['line_width']['shap'] = 3
mp_custom.config['line_width']['glm'] = 3

print("Custom configuration applied!")

In [ ]:
# Plot with custom styling
mp_custom.univariate_plot(
    var_name='age',
    shap=True,
    glm=True,
    actuals=True,
    weight=True,
    plot_name='Age - Custom Styled Plot'
)

## 8. Plot Engines

MintyPython supports two plot engines: Bokeh (interactive) and Matplotlib (static).

### 8.1 Bokeh Engine (Interactive)

In [ ]:
# Bokeh is the default engine - produces interactive HTML plots
mp.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Bokeh',
    plot_name='Years Licensed - Bokeh (Interactive)'
)

### 8.2 Matplotlib Engine (Static)

In [ ]:
# Matplotlib produces static plots - useful for reports and PDFs
mp.univariate_plot(
    var_name='years_licensed',
    shap=True,
    glm=True,
    weight=True,
    engine='Matplotlib',
    plot_name='Years Licensed - Matplotlib (Static)'
)

## Summary

This demo covered:

1. **Data Generation**: Creating synthetic insurance data with realistic feature relationships
2. **Model Training**: XGBoost with Poisson objective for claim frequency modeling
3. **MintyPython Initialization**: Setting up the library with models, data, and GLM comparisons
4. **Univariate Plots**: Various ways to visualize single-feature effects
5. **Bivariate Plots**: Visualizing feature interactions
6. **Model Comparison**: Comparing multiple GBM models side by side
7. **Customization**: Changing colors, labels, and styling
8. **Plot Engines**: Using Bokeh for interactive plots and Matplotlib for static output

For more details, see the MintyPython documentation and the CLAUDE.md file in the repository root.